In [1]:
from datasets import load_dataset
import pandas as pd
import numpy as np

RANDOM_STATE = 42

# Load Linguistic Bias dataset
ds = load_dataset("mediabiasgroup/mbib-base")
df = ds["linguistic_bias"].to_pandas()

# Create a copy for cleaning
clean_df = df.copy()

# Normalize text for duplicate detection
clean_df["normalized_text"] = (
    clean_df["text"]
    .fillna("")
    .str.lower()
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
)

# Identify conflicting labels
label_counts = (
    clean_df.groupby("normalized_text")["label"]
    .nunique()
)

conflicting_keys = label_counts[
    label_counts > 1
].index

# Remove conflicting text groups
clean_df = clean_df[
    ~clean_df["normalized_text"].isin(conflicting_keys)
].copy()

# Remove consistent normalized duplicates
clean_df = clean_df.drop_duplicates(
    subset=["normalized_text"],
    keep="first"
).copy()

# Validate dataset
assert clean_df["label"].isin([0, 1]).all()
assert clean_df["normalized_text"].is_unique
assert clean_df["text"].notna().all()

print("Original dataset:", df.shape)
print("Cleaned dataset:", clean_df.shape)

display(clean_df["label"].value_counts())

Original dataset: (401862, 4)
Cleaned dataset: (400657, 5)


label
0    200426
1    200231
Name: count, dtype: int64

In [3]:
from sklearn.model_selection import StratifiedGroupKFold

# Group samples sharing the same ID in Source 75.
# Other sources use a unique group per row.
groups = pd.Series(
    np.where(
        clean_df["dataset_id"] == 75,
        clean_df["dataset_id"].astype(str) + "_" + clean_df["id"].astype(str),
        "row_" + clean_df.index.astype(str)
    ),
    index=clean_df.index
)

# First split: approximately 80% train / 20% temporary
cv_outer = StratifiedGroupKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

train_idx, temp_idx = next(
    cv_outer.split(
        clean_df["text"],
        clean_df["label"],
        groups
    )
)

train_df = clean_df.iloc[train_idx].copy()
temp_df = clean_df.iloc[temp_idx].copy()

# Second split: temporary data into validation and test
temp_groups = groups.iloc[temp_idx]

cv_inner = StratifiedGroupKFold(
    n_splits=2,
    shuffle=True,
    random_state=42
)

val_idx, test_idx = next(
    cv_inner.split(
        temp_df["text"],
        temp_df["label"],
        temp_groups
    )
)

val_df = temp_df.iloc[val_idx].copy()
test_df = temp_df.iloc[test_idx].copy()

# Check results
for name, subset in [
    ("Train", train_df),
    ("Validation", val_df),
    ("Test", test_df)
]:
    print(
        name,
        len(subset),
        f"({len(subset)/len(clean_df)*100:.2f}%)",
        subset["label"].value_counts(normalize=True).round(3).to_dict())

Train 320422 (79.97%) {0: 0.5, 1: 0.5}
Validation 40144 (10.02%) {1: 0.5, 0: 0.5}
Test 40091 (10.01%) {0: 0.501, 1: 0.499}


In [4]:
# Data Leakage Check
# Check text overlap across splits

def check_overlap(col):
    train = set(train_df[col])
    val = set(val_df[col])
    test = set(test_df[col])

    print(f"\n{col} overlap:")
    print("Train / Validation:", len(train & val))
    print("Train / Test:", len(train & test))
    print("Validation / Test:", len(val & test))


check_overlap("normalized_text")

# Check related IDs in Source 75
train_ids = set(train_df.loc[train_df["dataset_id"] == 75, "id"])
val_ids = set(val_df.loc[val_df["dataset_id"] == 75, "id"])
test_ids = set(test_df.loc[test_df["dataset_id"] == 75, "id"])

print("\nSource 75 ID overlap:")
print("Train / Validation:", len(train_ids & val_ids))
print("Train / Test:", len(train_ids & test_ids))
print("Validation / Test:", len(val_ids & test_ids))

# Verify all records are accounted for
assert len(train_df) + len(val_df) + len(test_df) == len(clean_df)


normalized_text overlap:
Train / Validation: 0
Train / Test: 0
Validation / Test: 0

Source 75 ID overlap:
Train / Validation: 0
Train / Test: 0
Validation / Test: 0


In [5]:
print("Train shape:", train_df.shape)
print("Validation shape:", val_df.shape)

print("\nTrain labels:")
print(train_df["label"].value_counts())

print("\nValidation labels:")
print(val_df["label"].value_counts())

print("\nMissing text values:")
print("Train:", train_df["text"].isna().sum())
print("Validation:", val_df["text"].isna().sum())

Train shape: (320422, 5)
Validation shape: (40144, 5)

Train labels:
label
0    160286
1    160136
Name: count, dtype: int64

Validation labels:
label
1    20080
0    20064
Name: count, dtype: int64

Missing text values:
Train: 0
Validation: 0


In [6]:
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import FeatureUnion

# Small, balanced training sample
experiment_df, _ = train_test_split(
    train_df,
    train_size=20000,
    stratify=train_df["label"],
    random_state=42
)

word_vectorizer = TfidfVectorizer(
    analyzer="word",
    ngram_range=(1, 2),
    max_features=20000,
    min_df=2,
    sublinear_tf=True
)

char_vectorizer = TfidfVectorizer(
    analyzer="char",
    ngram_range=(3, 5),
    max_features=20000,
    min_df=2,
    sublinear_tf=True
)

combined_vectorizer = FeatureUnion([
    ("word", word_vectorizer),
    ("char", char_vectorizer)
])

print("Experiment samples:", len(experiment_df))
print("Labels:")
print(experiment_df["label"].value_counts())
print("Vectorizer ready!")

Experiment samples: 20000
Labels:
label
0    10005
1     9995
Name: count, dtype: int64
Vectorizer ready!


In [7]:
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
import time

combined_pipeline = Pipeline([
    ("vectorizer", combined_vectorizer),
    ("classifier", LogisticRegression(
        max_iter=1000,
        random_state=42
    ))
])

start = time.perf_counter()

combined_pipeline.fit(
    experiment_df["text"],
    experiment_df["label"]
)

training_time = time.perf_counter() - start

val_probs = combined_pipeline.predict_proba(
    val_df["text"]
)[:, 1]

combined_auc = roc_auc_score(
    val_df["label"],
    val_probs
)

print(f"Training time: {training_time:.2f} seconds")
print(f"Validation ROC-AUC: {combined_auc:.4f}")

Training time: 5.46 seconds
Validation ROC-AUC: 0.6479


In [8]:
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
import time

combined_pipeline = Pipeline([
    ("vectorizer", combined_vectorizer),
    ("classifier", LogisticRegression(
        max_iter=1000,
        random_state=42
    ))
])

start = time.perf_counter()

combined_pipeline.fit(
    experiment_df["text"],
    experiment_df["label"]
)

training_time = time.perf_counter() - start

val_probs = combined_pipeline.predict_proba(
    val_df["text"]
)[:, 1]

combined_auc = roc_auc_score(
    val_df["label"],
    val_probs
)

print(f"Training time: {training_time:.2f} seconds")
print(f"Validation ROC-AUC: {combined_auc:.4f}")

Training time: 5.54 seconds
Validation ROC-AUC: 0.6479


In [9]:
from sklearn.base import clone

word_pipeline = Pipeline([
    ("vectorizer", clone(word_vectorizer)),
    ("classifier", LogisticRegression(
        max_iter=1000,
        random_state=42
    ))
])

start = time.perf_counter()

word_pipeline.fit(
    experiment_df["text"],
    experiment_df["label"]
)

word_probs = word_pipeline.predict_proba(
    val_df["text"]
)[:, 1]

word_auc = roc_auc_score(
    val_df["label"],
    word_probs
)

print(f"Word-only ROC-AUC: {word_auc:.4f}")
print(f"Word + Character ROC-AUC: {combined_auc:.4f}")
print(f"Difference: {combined_auc - word_auc:+.4f}")
print(f"Time: {time.perf_counter() - start:.2f} seconds")

Word-only ROC-AUC: 0.6343
Word + Character ROC-AUC: 0.6479
Difference: +0.0136
Time: 2.34 seconds


In [10]:
experiment_40k_df, _ = train_test_split(
    train_df,
    train_size=40000,
    stratify=train_df["label"],
    random_state=42
)

combined_40k_pipeline = clone(combined_pipeline)

start = time.perf_counter()

combined_40k_pipeline.fit(
    experiment_40k_df["text"],
    experiment_40k_df["label"]
)

probs_40k = combined_40k_pipeline.predict_proba(
    val_df["text"]
)[:, 1]

auc_40k = roc_auc_score(
    val_df["label"],
    probs_40k
)

print(f"Training samples: {len(experiment_40k_df):,}")
print(f"Training time: {time.perf_counter() - start:.2f} seconds")
print(f"Validation ROC-AUC: {auc_40k:.4f}")

Training samples: 40,000
Training time: 21.20 seconds
Validation ROC-AUC: 0.6622


In [11]:
experiment_100k_df, _ = train_test_split(
    train_df,
    train_size=100000,
    stratify=train_df["label"],
    random_state=42
)

combined_100k_pipeline = clone(combined_pipeline)

start = time.perf_counter()

combined_100k_pipeline.fit(
    experiment_100k_df["text"],
    experiment_100k_df["label"]
)

probs_100k = combined_100k_pipeline.predict_proba(
    val_df["text"]
)[:, 1]

auc_100k = roc_auc_score(
    val_df["label"],
    probs_100k
)

print(f"Training samples: {len(experiment_100k_df):,}")
print(f"Training time: {time.perf_counter() - start:.2f} seconds")
print(f"Validation ROC-AUC: {auc_100k:.4f}")

Training samples: 100,000
Training time: 40.87 seconds
Validation ROC-AUC: 0.6765


In [12]:
experiment_200k_df, _ = train_test_split(
    train_df,
    train_size=200000,
    stratify=train_df["label"],
    random_state=42
)

combined_200k_pipeline = clone(combined_pipeline)

start = time.perf_counter()

combined_200k_pipeline.fit(
    experiment_200k_df["text"],
    experiment_200k_df["label"]
)

probs_200k = combined_200k_pipeline.predict_proba(
    val_df["text"]
)[:, 1]

auc_200k = roc_auc_score(
    val_df["label"],
    probs_200k
)

print(f"Training samples: {len(experiment_200k_df):,}")
print(f"Training time: {time.perf_counter() - start:.2f} seconds")
print(f"Validation ROC-AUC: {auc_200k:.4f}")

Training samples: 200,000
Training time: 74.16 seconds
Validation ROC-AUC: 0.6867


In [13]:
combined_full_pipeline = clone(combined_pipeline)

start = time.perf_counter()

combined_full_pipeline.fit(
    train_df["text"],
    train_df["label"]
)

probs_full = combined_full_pipeline.predict_proba(
    val_df["text"]
)[:, 1]

auc_full = roc_auc_score(
    val_df["label"],
    probs_full
)

print(f"Training samples: {len(train_df):,}")
print(f"Training time: {time.perf_counter() - start:.2f} seconds")
print(f"Validation ROC-AUC: {auc_full:.4f}")
print(f"Improvement over original LR: {auc_full - 0.6838:+.4f}")

Training samples: 320,422
Training time: 106.80 seconds
Validation ROC-AUC: 0.6950
Improvement over original LR: +0.0112
